# DSA4213 — N-gram: 30-configuration search


## Run in Colab
1. Choose a **CPU runtime** and check `PROJECT_ROOT` in Section 2.
2. Leave `TRAIN_IF_MISSING=True` and `EVALUATE_TEST=False`, then **Run all**. Existing saved preprocessing outputs are reused.
3. Review Section 8. Once the validation-selected configuration is final, set `EVALUATE_TEST=True` and **Run all** again. The model reloads without fitting/tuning again, and test results are exported.

## Comparison
Five k smoothing values times six interpolating weight triples give 30 candidates. Weights are ordered unigram, bigram, trigram. The data, vocabulary, targets, probability formula, context handling and generation workload are unchanged for each candidate.

Only validation is used for selection within each model. Test set is used to compare across the 4 models. 



## 1. Dependencies

In [13]:
%pip -q install numpy pandas psutil threadpoolctl

## 2. Drive, paths and run controls

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, time, hashlib, pickle, shutil, platform, subprocess, gc
import random, importlib, importlib.metadata as metadata, uuid
from datetime import datetime, timezone
from contextlib import contextmanager
import threading
import numpy as np
import pandas as pd
import psutil
from threadpoolctl import threadpool_limits, threadpool_info
from IPython.display import display

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/DSA4213_Assignment1')
LOCAL_ROOT = Path('/content/dsa4213_ngram_work')
TRAIN_IF_MISSING = True # if not existing model, train will be run
EVALUATE_TEST = True # evaluates the model on test set
RUN_BENCHMARK = True
SEED = 42
CPU_THREADS = 1
random.seed(SEED)
np.random.seed(SEED)
THREAD_LIMITER = threadpool_limits(limits=CPU_THREADS)
SESSION_START = time.perf_counter()
SESSION_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:6]
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

def write_json(path, obj):
    path = Path(path)
    temp = path.with_name(path.name + '.tmp')
    temp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    temp.replace(path)

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

def object_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True).encode()).hexdigest()

print('Project:', PROJECT_ROOT)
print('This session:', SESSION_ID)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/DSA4213_Assignment1
This session: 20261014T154830Z_185bd5


## 3. Load and verify the shared data contract

In [15]:
common_path = PROJECT_ROOT / 'configs/common.json'
if not common_path.exists():
    raise FileNotFoundError(f'Run notebook 01 to completion first, or fix PROJECT_ROOT: {common_path}')
common = json.loads(common_path.read_text())
DATA_DIR = PROJECT_ROOT / common['processed_dir_relative']
completion = json.loads((DATA_DIR / 'COMPLETE.json').read_text())
assert completion['data_version'] == common['data_version']
for name, expected in completion['sha256'].items():
    assert file_hash(DATA_DIR / name) == expected, f'Checksum mismatch: {name}'
assert json.loads((DATA_DIR / 'config.json').read_text()) == common
assert common['special_ids'] == {'PAD': 0, 'UNK': 1, 'BOS': 2, 'EOS': 3}
assert common['reset_state_each_block'] is True
PAD, UNK, BOS, EOS = 0, 1, 2, 3
V = common['vocab_size']
assert common['predicted_ids'] == [i for i in range(V) if i not in (PAD, BOS)]
LOCAL_DATA = LOCAL_ROOT / common['data_version']
LOCAL_DATA.mkdir(parents=True, exist_ok=True)
for name in completion['sha256']:
    if name.endswith('_blocks.npz') or name in ('vocab.json', 'preprocessing.py', 'generation_prompts.json'):
        dest = LOCAL_DATA / name
        if not dest.exists() or file_hash(dest) != completion['sha256'][name]:
            shutil.copy2(DATA_DIR / name, dest)
        assert file_hash(dest) == completion['sha256'][name]
vocab = json.loads((LOCAL_DATA / 'vocab.json').read_text())
id_to_token, token_to_id = vocab['id_to_token'], vocab['token_to_id']
assert len(id_to_token) == V
assert all(token_to_id[token] == i for i, token in enumerate(id_to_token))
blocks, target_audit = {}, []
for split in ('train', 'validation', 'test'):
    with np.load(LOCAL_DATA / f'{split}_blocks.npz', allow_pickle=False) as a:
        x, y = a['x'], a['y']
    manifest = pd.read_csv(DATA_DIR / f'{split}_manifest.csv')
    assert x.shape == y.shape == (len(manifest), common['block_length'])
    assert np.issubdtype(x.dtype, np.integer) and np.issubdtype(y.dtype, np.integer)
    lengths = manifest.valid_length.to_numpy(dtype=int)
    assert np.all((lengths >= 1) & (lengths <= x.shape[1]))
    assert np.array_equal(manifest.block_id.to_numpy(), np.arange(len(x)))
    valid = np.arange(x.shape[1])[None, :] < lengths[:, None]
    assert np.array_equal(y != PAD, valid) and np.array_equal(x != PAD, valid)
    assert x.min() >= 0 and y.min() >= 0 and x.max() < V and y.max() < V
    assert not np.any(y == BOS)
    pair_mask = np.arange(x.shape[1]-1)[None, :] < (lengths-1)[:, None]
    assert np.array_equal(x[:, 1:][pair_mask], y[:, :-1][pair_mask])
    assert np.all(x[manifest.offset.to_numpy() == 0, 0] == BOS)
    targets = y[valid].astype('<i4')
    target_audit.append({'split': split, 'blocks': len(x), 'target_count': len(targets),
        'target_ids_sha256': hashlib.sha256(targets.tobytes()).hexdigest(),
        'blocks_sha256': completion['sha256'][f'{split}_blocks.npz'],
        'manifest_sha256': completion['sha256'][f'{split}_manifest.csv']})
    blocks[split] = (x, y)
DATA_HASH = file_hash(DATA_DIR / 'COMPLETE.json')
VOCAB_HASH = completion['sha256']['vocab.json']
data_summary = pd.read_csv(DATA_DIR / 'data_summary.csv')
display(data_summary.round(3))
display(pd.DataFrame(target_audit)[['split', 'blocks', 'target_count']])
print('Data version:', common['data_version'], '| vocabulary:', V, '| output alphabet:', V-2)

,split,raw_rows,retained_rows,removed_rows,lexical_tokens,unknown_tokens,unknown_pct,median_row_tokens,p95_row_tokens,max_row_tokens,vocab_size
0,train,36718,21441,15277,2115237,160711,7.598,90.0,252.0,746,10000
1,validation,3760,2297,1463,221408,20450,9.236,86.0,245.0,437,10000
2,test,4358,2792,1566,249905,25189,10.079,76.0,245.0,497,10000


,split,blocks,target_count
0,train,46109,2136678
1,validation,4845,223705
2,test,5652,252697


Data version: beacb3de63dc4261 | vocabulary: 10000 | output alphabet: 9998


## 4. Model definition and probability formula

In [16]:
MODEL_SOURCE = r'''
"""DSA4213 sparse interpolated count LM. Only NumPy and Python stdlib needed."""
from collections import Counter
import math
import numpy as np

class InterpolatedNGram:
    def __init__(self, vocab_size, predicted_ids, pad=0, bos=2):
        self.vocab_size = int(vocab_size)
        self.allowed = np.asarray(predicted_ids, dtype=np.int64)
        self.pad, self.bos = pad, bos
        self.K = len(self.allowed)
        self.unigrams = Counter()
        self.bigrams, self.trigrams = {}, {}
        self.bigram_totals, self.trigram_totals = Counter(), Counter()
        self.total = 0
        self.k, self.weights = None, None

    def fit(self, x, y):
        if self.total:
            raise RuntimeError('Fit a fresh instance; do not count the same corpus twice.')
        for inputs, targets in zip(x, y):
            for t, target in enumerate(targets):
                w = int(target)
                if w == self.pad:
                    continue
                h1 = int(inputs[t])
                self.unigrams[w] += 1
                self.bigrams.setdefault(h1, Counter())[w] += 1
                self.bigram_totals[h1] += 1
                # At t=0 the trigram component later reuses the bigram;
                # do not pretend that a second history token exists.
                if t >= 1:
                    h2 = (int(inputs[t-1]), h1)
                    self.trigrams.setdefault(h2, Counter())[w] += 1
                    self.trigram_totals[h2] += 1
                self.total += 1
        if not self.total:
            raise ValueError('Empty training targets.')
        return self

    def set_smoothing(self, k, weights):
        if k <= 0 or len(weights) != 3 or min(weights) < 0 or not np.isclose(sum(weights), 1):
            raise ValueError('Require k>0 and three nonnegative weights summing to 1.')
        self.k, self.weights = float(k), tuple(float(v) for v in weights)
        return self

    def counts_for_targets(self, x, y):
        """One row per valid target, in saved block-major order; no future inputs."""
        n = int(np.count_nonzero(y != self.pad))
        counts = np.empty((n, 3), dtype=np.float64)
        totals = np.empty_like(counts)
        at = 0
        for inputs, targets in zip(x, y):
            for t, target in enumerate(targets):
                w = int(target)
                if w == self.pad:
                    continue
                h1 = int(inputs[t])
                c1, d1 = self.unigrams.get(w, 0), self.total
                c2 = self.bigrams.get(h1, {}).get(w, 0)
                d2 = self.bigram_totals.get(h1, 0)
                if t:
                    h2 = (int(inputs[t-1]), h1)
                    c3 = self.trigrams.get(h2, {}).get(w, 0)
                    d3 = self.trigram_totals.get(h2, 0)
                else:
                    c3, d3 = c2, d2
                counts[at] = c1, c2, c3
                totals[at] = d1, d2, d3
                at += 1
        return counts, totals

    def score_counts(self, counts, totals, k=None, weights=None):
        k = self.k if k is None else k
        weights = self.weights if weights is None else weights
        probabilities = ((counts + k) / (totals + k * self.K)) @ np.asarray(weights)
        if not len(probabilities) or not np.all(np.isfinite(probabilities)) or np.any(probabilities <= 0):
            raise ValueError('Invalid probability or empty scored split.')
        nll = float(-np.log(probabilities).sum(dtype=np.float64))
        ce = nll / len(probabilities)
        return {'nll_sum': nll, 'token_count': len(probabilities), 'ce': ce,
                'ppl': math.exp(ce) if ce < math.log(np.finfo(float).max) else None}

    def distribution(self, history):
        """Full vocabulary vector, with zero PAD/BOS probability."""
        if self.k is None:
            raise RuntimeError('Select smoothing before prediction.')
        history = tuple(int(v) for v in history)
        tables = [self.unigrams]
        totals = [self.total]
        if history:
            h1 = history[-1]
            tables.append(self.bigrams.get(h1, {}))
            totals.append(self.bigram_totals.get(h1, 0))
        else:
            tables.append(tables[0]); totals.append(totals[0])
        if len(history) >= 2:
            h2 = history[-2:]
            tables.append(self.trigrams.get(h2, {}))
            totals.append(self.trigram_totals.get(h2, 0))
        else:
            tables.append(tables[1]); totals.append(totals[1])
        out = np.zeros(self.vocab_size, dtype=np.float64)
        for weight, table, total in zip(self.weights, tables, totals):
            denominator = total + self.k * self.K
            out[self.allowed] += weight * self.k / denominator
            if table:
                ids = np.fromiter(table.keys(), dtype=np.int64, count=len(table))
                values = np.fromiter(table.values(), dtype=np.float64, count=len(table))
                out[ids] += weight * values / denominator
        return out

    def entry_counts(self):
        return {'unigram_entries': len(self.unigrams),
                'bigram_entries': sum(map(len, self.bigrams.values())),
                'trigram_entries': sum(map(len, self.trigrams.values())),
                'bigram_contexts': len(self.bigrams), 'trigram_contexts': len(self.trigrams)}


def generate(model, prompt_ids, *, bos=2, eos=3, forbidden=(0, 1, 2),
             max_new_tokens=50, decoding='greedy', temperature=0.8, top_k=50,
             seed=123, stop_at_eos=True):
    if decoding not in ('greedy', 'sampling'):
        raise ValueError('Unknown decoder.')
    rng = np.random.default_rng(seed)
    prefix = [bos] + list(prompt_ids)
    generated = []
    for _ in range(max_new_tokens):
        probabilities = model.distribution(prefix[-2:])
        probabilities[list(forbidden)] = 0.0
        if decoding == 'greedy':
            next_id = int(np.argmax(probabilities))  # lowest ID breaks ties
        else:
            eligible = np.flatnonzero(probabilities > 0)
            # Stable descending sort: lower token ID wins exact ties.
            eligible = eligible[np.argsort(-probabilities[eligible], kind='stable')][:top_k]
            logits = np.log(probabilities[eligible]) / temperature
            weights = np.exp(logits - logits.max())
            weights /= weights.sum()
            next_id = int(rng.choice(eligible, p=weights))
        generated.append(next_id)
        prefix.append(next_id)
        if stop_at_eos and next_id == eos:
            break
    return {'token_ids': generated, 'generated_length': len(generated),
            'stop_reason': 'eos' if stop_at_eos and generated and generated[-1] == eos else 'length_cap',
            'eos_count': generated.count(eos)}
'''

MODULE_HASH = hashlib.sha256(MODEL_SOURCE.encode()).hexdigest()
MODULE_NAME = 'ngram_core_' + MODULE_HASH[:12]
MODULE_PATH = LOCAL_ROOT / (MODULE_NAME + '.py')
MODULE_PATH.write_text(MODEL_SOURCE, encoding='utf-8')
sys.path.insert(0, str(LOCAL_ROOT))
core = importlib.import_module(MODULE_NAME)
InterpolatedNGram, generate = core.InterpolatedNGram, core.generate

## 5. Correctness checks before fitting

In [17]:
toy_x = np.array([[BOS, 4, 5, PAD], [6, 4, PAD, PAD]], dtype=np.int32)
toy_y = np.array([[4, 5, EOS, PAD], [4, EOS, PAD, PAD]], dtype=np.int32)
toy = InterpolatedNGram(8, [1, 3, 4, 5, 6, 7]).fit(toy_x, toy_y).set_smoothing(0.1, (0.1, 0.3, 0.6))
assert toy.total == 5 and toy.unigrams[4] == 2 and toy.unigrams[EOS] == 2
assert sum(toy.trigram_totals.values()) == 3
for h in ([], [BOS], [BOS, 4], [7, 7], [6]):
    p = toy.distribution(h)
    assert np.isclose(p.sum(), 1.0) and np.all(p >= 0)
    assert p[PAD] == p[BOS] == 0
c, d = toy.counts_for_targets(toy_x, toy_y)
fast = toy.score_counts(c, d)
manual_nll = 0.0
for inputs, targets in zip(toy_x, toy_y):
    for t, w in enumerate(targets):
        if w != PAD:
            manual_nll -= np.log(toy.distribution(inputs[:t+1])[w])
assert np.isclose(manual_nll, fast['nll_sum'])
c2, d2 = toy.counts_for_targets(np.pad(toy_x, ((0,0),(0,3))), np.pad(toy_y, ((0,0),(0,3))))
assert np.array_equal(c, c2) and np.array_equal(d, d2)
# At block start, trigram uses exactly the one-token-history component.
assert c[0, 1] == c[0, 2] and d[0, 1] == d[0, 2]
assert c[3, 1] == c[3, 2] and d[3, 1] == d[3, 2]
print('All model correctness checks passed.')

All model correctness checks passed.


## 6. Freeze settings and capture the machine
The 30 trials combine five smoothing values and six weight triples, including all original candidates. The first trial in the saved order wins an exact validation tie. Count fitting is one pass; neural epochs, optimizer updates and neural parameter counts are **not applicable**.



In [18]:
SETTINGS = {
    'part': 'I', 'model': 'interpolated_unigram_bigram_trigram', 'seed': SEED,
    'k_grid': [0.00001, 0.0001, 0.001, 0.01, 0.1],
    'weight_grid': [[0.05, 0.15, 0.80], [0.10, 0.30, 0.60],
                    [0.20, 0.30, 0.50], [0.10, 0.60, 0.30],
                    [0.30, 0.50, 0.20], [0.60, 0.30, 0.10]],
    'selection': 'minimum validation CE; candidate order breaks exact ties',
    'short_history': 'trigram reuses bigram at first block position',
    'device': 'cpu', 'CPU_threads': CPU_THREADS, 'probability_precision': 'float64',
    'count_storage': 'Python integer sparse dictionaries', 'training_passes': 1,
    'data_hash': DATA_HASH, 'vocab_hash': VOCAB_HASH, 'module_hash': MODULE_HASH,
    'data_version': common['data_version'], 'block_length': common['block_length'],
    'generation': {'max_new_tokens': 50, 'temperature': 0.8, 'top_k': 50,
                   'seed_per_prompt': 123, 'forbidden_ids': [PAD, UNK, BOS]},
    'benchmark': {'warmups': 1, 'repeats': 3, 'batch_size': 1, 'decoding': 'greedy',
                  'tokens_per_prompt': 50, 'stop_at_eos': False, 'CPU_threads': CPU_THREADS}
}
N_CONFIGURATIONS = len(SETTINGS['k_grid']) * len(SETTINGS['weight_grid'])
assert N_CONFIGURATIONS == 30
print('Validation candidates:', N_CONFIGURATIONS)
CONFIG_HASH = object_hash(SETTINGS)
RUN_ID = 'I_ngram_' + CONFIG_HASH[:12]
RUN_DIR = PROJECT_ROOT / 'runs' / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
SESSION_DIR = RUN_DIR / 'sessions' / SESSION_ID
SESSION_DIR.mkdir(parents=True, exist_ok=False)
RESULT_DIR = PROJECT_ROOT / 'results' / RUN_ID / SESSION_ID
RESULT_DIR.mkdir(parents=True, exist_ok=False)
if (RUN_DIR / 'config.json').exists():
    assert json.loads((RUN_DIR / 'config.json').read_text()) == SETTINGS
else:
    write_json(RUN_DIR / 'config.json', SETTINGS)
    write_json(RUN_DIR / 'common.json', common)
    write_json(RUN_DIR / 'data_complete.json', completion)
    shutil.copy2(DATA_DIR / 'source.json', RUN_DIR / 'data_source.json')
    shutil.copy2(LOCAL_DATA / 'vocab.json', RUN_DIR / 'vocab.json')
    shutil.copy2(LOCAL_DATA / 'preprocessing.py', RUN_DIR / 'preprocessing.py')
    shutil.copy2(MODULE_PATH, RUN_DIR / MODULE_PATH.name)
write_json(SESSION_DIR / 'target_audit.json', target_audit)

def command_output(command):
    try:
        p = subprocess.run(command, capture_output=True, text=True, timeout=15)
        return p.stdout.strip() if p.returncode == 0 else None
    except (OSError, subprocess.TimeoutExpired):
        return None

cpu_info = command_output(['lscpu'])
environment = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(), 'session_id': SESSION_ID,
    'python': sys.version, 'platform': platform.platform(), 'machine': platform.machine(),
    'cpu_model': next((l.split(':',1)[1].strip() for l in (cpu_info or '').splitlines()
                       if l.startswith('Model name:')), platform.processor() or None),
    'lscpu': cpu_info, 'logical_cpus': psutil.cpu_count(), 'physical_cpus': psutil.cpu_count(logical=False),
    'cpu_affinity': psutil.Process().cpu_affinity() if hasattr(psutil.Process(), 'cpu_affinity') else None,
    'ram_total_gib': psutil.virtual_memory().total / 2**30,
    'ram_available_gib': psutil.virtual_memory().available / 2**30,
    'gpu_inventory': command_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader']),
    'actual_model_device': 'cpu', 'CPU_threads_requested': CPU_THREADS,
    'threadpools': threadpool_info(), 'precision': 'float64 probabilities; Python integer counts',
    'packages': {}, 'PYTHONHASHSEED': os.environ.get('PYTHONHASHSEED'),
    'determinism_note': 'No dependence on set iteration order; no claim of cross-platform bitwise reproducibility.'
}
for package in ['numpy', 'pandas', 'psutil', 'threadpoolctl', 'torch', 'datasets', 'huggingface_hub', 'gensim', 'scikit-learn']:
    try:
        environment['packages'][package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        environment['packages'][package] = None
write_json(SESSION_DIR / 'environment.json', environment)
(SESSION_DIR / 'environment-full.txt').write_text(command_output([sys.executable, '-m', 'pip', 'freeze']) or 'Unavailable')
(SESSION_DIR / 'requirements-ngram-lock.txt').write_text(''.join(
    f'{p}=={environment["packages"][p]}\n' for p in ['numpy', 'pandas', 'psutil', 'threadpoolctl']))
prep_env = json.loads((DATA_DIR / 'environment.json').read_text())
version_comparison = pd.DataFrame([{'package': p, 'preparation': old,
    'current': environment['packages'].get(p), 'matches': old == environment['packages'].get(p)}
    for p, old in prep_env.get('packages', {}).items()])
version_comparison.to_csv(SESSION_DIR / 'version_comparison.csv', index=False)
display(version_comparison)
print('Version differences are logged; data content and tokenizer hashes must still match.')
print('CPU:', environment['cpu_model'], '| RAM GiB:', round(environment['ram_total_gib'], 2))
print('Model device: CPU | numerical-library threads:', CPU_THREADS)
print('Run:', RUN_DIR)

@contextmanager
def measured():
    process = psutil.Process()
    stop = threading.Event()
    report = {'rss_before_mib': process.memory_info().rss / 2**20}
    peak = [report['rss_before_mib']]
    def sample():
        while not stop.wait(0.05):
            peak[0] = max(peak[0], process.memory_info().rss / 2**20)
    worker = threading.Thread(target=sample, daemon=True)
    worker.start()
    wall, cpu = time.perf_counter(), time.process_time()
    try:
        yield report
    finally:
        report['seconds'] = time.perf_counter() - wall
        report['process_cpu_seconds'] = time.process_time() - cpu
        stop.set(); worker.join()
        report['rss_after_mib'] = process.memory_info().rss / 2**20
        report['sampled_peak_rss_mib'] = max(peak[0], report['rss_after_mib'])
        report['sampled_peak_increment_mib'] = report['sampled_peak_rss_mib'] - report['rss_before_mib']

io_log = []
def save_pickle_verified(path, obj):
    start = time.perf_counter()
    temp = path.with_name(path.name + '.tmp')
    with temp.open('wb') as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    temp.replace(path)
    write_json(path.with_suffix('.sha256.json'), {'sha256': file_hash(path)})
    io_log.append({'operation': 'serialize_and_save', 'filename': path.name, 'seconds': time.perf_counter()-start})

def load_pickle_verified(path):
    # Only load checkpoints created by this notebook in your own project.
    marker = path.with_suffix('.sha256.json')
    if not marker.exists():
        raise RuntimeError(f'Incomplete checkpoint: {path}. Inspect/rename it, then retry.')
    assert file_hash(path) == json.loads(marker.read_text())['sha256']
    start = time.perf_counter()
    with path.open('rb') as f:
        obj = pickle.load(f)
    io_log.append({'operation': 'load', 'filename': path.name, 'seconds': time.perf_counter()-start})
    return obj

Validation candidates: 30


,package,preparation,current,matches
0,datasets,4.8.5,4.8.5,True
1,huggingface_hub,1.33.0,1.33.0,True
2,numpy,2.1.3,2.1.3,True
3,pandas,2.2.3,2.2.3,True


Version differences are logged; data content and tokenizer hashes must still match.
CPU: Intel(R) Xeon(R) CPU @ 2.20GHz | RAM GiB: 12.67
Model device: CPU | numerical-library threads: 1
Run: /content/drive/MyDrive/DSA4213_Assignment1/runs/I_ngram_05cfd29bc588


## 7. Fit count tables, or reload a completed fit

In [19]:
COUNTS_PATH = RUN_DIR / 'counts.pkl'
MODEL_PATH = RUN_DIR / 'model.pkl'
if MODEL_PATH.exists():
    checkpoint = load_pickle_verified(MODEL_PATH)
    model = checkpoint['model']
    fit_info = checkpoint['fit_info']
    print('Loaded selected model; fitting and tuning will be skipped.')
elif COUNTS_PATH.exists():
    checkpoint = load_pickle_verified(COUNTS_PATH)
    model, fit_info = checkpoint['model'], checkpoint['fit_info']
    print('Loaded counts; continue to validation selection.')
else:
    if not TRAIN_IF_MISSING:
        raise FileNotFoundError('No saved model. Set TRAIN_IF_MISSING=True to fit it.')
    model = InterpolatedNGram(V, common['predicted_ids'], PAD, BOS)
    gc.collect()
    with measured() as fit_measurement:
        model.fit(*blocks['train'])
    fit_info = {**fit_measurement, 'training_environment': environment,
                'training_targets': model.total, 'training_passes': 1}
    checkpoint = {'model': model, 'fit_info': fit_info, 'config_hash': CONFIG_HASH,
                  'data_hash': DATA_HASH, 'vocab_hash': VOCAB_HASH, 'module_hash': MODULE_HASH}
    save_pickle_verified(COUNTS_PATH, checkpoint)
assert checkpoint['config_hash'] == CONFIG_HASH
assert checkpoint['data_hash'] == DATA_HASH and checkpoint['vocab_hash'] == VOCAB_HASH
assert checkpoint['module_hash'] == MODULE_HASH
assert model.total == target_audit[0]['target_count']
display(pd.DataFrame([model.entry_counts()]))
print(f"Original count-fitting time: {fit_info['seconds']:.3f} s")

Loaded selected model; fitting and tuning will be skipped.


,unigram_entries,bigram_entries,trigram_entries,bigram_contexts,trigram_contexts
0,9998,434188,1180777,9998,430518


Original count-fitting time: 15.850 s


## 8. Select smoothing on validation only

Cross-entropy = total negative natural-log probability / valid target count. 

Perplexity = `exp(CE)`.

In [20]:
if MODEL_PATH.exists():
    search_rows = checkpoint['search_rows']
    selection_info = checkpoint['selection_info']
    best = checkpoint['selected']
else:
    if not TRAIN_IF_MISSING:
        raise RuntimeError('Counts exist but selection is unfinished. Enable TRAIN_IF_MISSING to complete selection.')
    with measured() as selection_measurement:
        lookup_start = time.perf_counter()
        val_counts, val_totals = model.counts_for_targets(*blocks['validation'])
        lookup_seconds = time.perf_counter() - lookup_start
        search_rows = []
        for k in SETTINGS['k_grid']:
            for weights in SETTINGS['weight_grid']:
                start = time.perf_counter()
                metrics = model.score_counts(val_counts, val_totals, k, weights)
                search_rows.append({'candidate': len(search_rows), 'k': k,
                    'lambda1': weights[0], 'lambda2': weights[1], 'lambda3': weights[2],
                    **metrics, 'scoring_seconds': time.perf_counter()-start})
        best = min(search_rows, key=lambda row: (row['ce'], row['candidate']))
        model.set_smoothing(best['k'], (best['lambda1'], best['lambda2'], best['lambda3']))
    selection_info = {**selection_measurement, 'lookup_seconds': lookup_seconds,
                      'candidate_count': len(search_rows), 'selection_environment': environment}
    checkpoint.update({'model': model, 'search_rows': search_rows, 'selected': best,
                       'selection_info': selection_info})
    save_pickle_verified(MODEL_PATH, checkpoint)
    del val_counts, val_totals
assert len(search_rows) == N_CONFIGURATIONS
search_table = pd.DataFrame(search_rows)
search_table.to_csv(RUN_DIR / 'validation_search.csv', index=False)
write_json(RUN_DIR / 'training_cost.json', {'fit': fit_info, 'selection': selection_info,
    'fit_plus_selection_seconds': fit_info['seconds'] + selection_info['seconds'],
    'io_excluded': True})
display(search_table.sort_values(['ce', 'candidate']).round(5))
print('Selected k:', model.k, '| weights:', model.weights)
print(f"Fit: {fit_info['seconds']:.3f}s; selection: {selection_info['seconds']:.3f}s")
MODEL_HASH = file_hash(MODEL_PATH)
# Inference artifact: counts + selected settings + vocabulary + matching module.
# Excludes search history, environment, raw data and resumable count checkpoint.
INFERENCE_PATH = RUN_DIR / 'inference.pkl'
if not INFERENCE_PATH.exists():
    save_pickle_verified(INFERENCE_PATH, {'model': model, 'config_hash': CONFIG_HASH,
        'vocab_hash': VOCAB_HASH, 'data_hash': DATA_HASH, 'module_hash': MODULE_HASH})
INFERENCE_BYTES = INFERENCE_PATH.stat().st_size
PACKAGE_BYTES = INFERENCE_BYTES + (RUN_DIR / 'vocab.json').stat().st_size + (RUN_DIR / MODULE_PATH.name).stat().st_size + (RUN_DIR / 'preprocessing.py').stat().st_size
reloaded = load_pickle_verified(INFERENCE_PATH)['model']
for h in ([BOS], [BOS, min(4, V-1)], [EOS, EOS]):
    a, b = model.distribution(h), reloaded.distribution(h)
    assert np.array_equal(a, b) and np.isclose(a.sum(), 1.0)
assert generate(model, [4], max_new_tokens=10) == generate(reloaded, [4], max_new_tokens=10)
del reloaded
print(f'Inference model: {INFERENCE_BYTES/2**20:.3f} MiB; with vocab/source: {PACKAGE_BYTES/2**20:.3f} MiB')

,candidate,k,lambda1,lambda2,lambda3,nll_sum,token_count,ce,ppl,scoring_seconds
9,9,0.00010,0.10,0.60,0.3,1.118649e+06,223705,5.00056,148.49558,0.00930
10,10,0.00010,0.30,0.50,0.2,1.119290e+06,223705,5.00342,148.92132,0.00719
4,4,0.00001,0.30,0.50,0.2,1.119633e+06,223705,5.00495,149.15033,0.00865
15,15,0.00100,0.10,0.60,0.3,1.120028e+06,223705,5.00672,149.41380,0.00690
3,3,0.00001,0.10,0.60,0.3,1.123162e+06,223705,5.02073,151.52178,0.00686
16,16,0.00100,0.30,0.50,0.2,1.125375e+06,223705,5.03062,153.02821,0.00703
8,8,0.00010,0.20,0.30,0.5,1.126437e+06,223705,5.03537,153.75594,0.00943
2,2,0.00001,0.20,0.30,0.5,1.129621e+06,223705,5.04960,155.96054,0.00701
14,14,0.00100,0.20,0.30,0.5,1.133367e+06,223705,5.06635,158.59367,0.00748
7,7,0.00010,0.10,0.30,0.6,1.136072e+06,223705,5.07844,160.52286,0.00975


Selected k: 0.0001 | weights: (0.1, 0.6, 0.3)
Fit: 15.850s; selection: 1.246s
Inference model: 16.916 MiB; with vocab/source: 17.261 MiB


## 9. Evaluate the fixed model

In [21]:
metric_rows = []
for split in ('train', 'validation', 'test'):
    if split == 'test' and not EVALUATE_TEST:
        continue
    path = RUN_DIR / f'{split}_metrics.json'
    if path.exists():
        row = json.loads(path.read_text())
        assert row['checkpoint_sha256'] == MODEL_HASH
        assert row['data_hash'] == DATA_HASH
    else:
        with measured() as eval_measurement:
            c, d = model.counts_for_targets(*blocks[split])
            scores = model.score_counts(c, d)
        audit = next(a for a in target_audit if a['split'] == split)
        assert scores['token_count'] == audit['target_count']
        if split == 'validation':
            assert np.isclose(scores['ce'], best['ce'])
        row = {'run_id': RUN_ID, 'split': split, **scores, **eval_measurement,
               'data_hash': DATA_HASH, 'vocab_hash': VOCAB_HASH,
               'target_ids_sha256': audit['target_ids_sha256'],
               'checkpoint_sha256': MODEL_HASH, 'evaluation_session_id': SESSION_ID}
        write_json(path, row)
        del c, d
    metric_rows.append(row)
quality = pd.DataFrame(metric_rows)
quality.to_csv(RESULT_DIR / 'quality_by_split.csv', index=False)
display(quality[['split', 'token_count', 'nll_sum', 'ce', 'ppl', 'seconds']].round(5))
print('seconds = evaluation wall time from the original evaluation session, not reload time.')

,split,token_count,nll_sum,ce,ppl,seconds
0,train,2136678,6.867821e+06,3.21425,24.88467,14.27902
1,validation,223705,1.118649e+06,5.00056,148.49558,0.89272
2,test,252697,1.246436e+06,4.93253,138.73035,1.29509


seconds = evaluation wall time from the original evaluation session, not reload time.


## 10. Matched text generation

In [22]:
preprocess_namespace = {}
exec((LOCAL_DATA / 'preprocessing.py').read_text(), preprocess_namespace)
tokenize = preprocess_namespace['tokenize']
prompts = json.loads((LOCAL_DATA / 'generation_prompts.json').read_text())
assert prompts
for p in prompts:
    assert p['tokens'] == tokenize(p['text'])
    assert p['ids'] == [token_to_id.get(t, UNK) for t in p['tokens']]
    if UNK in p['ids']:
        print('Prompt contains UNK; retained consistently:', p['text'])
write_json(RUN_DIR / 'generation_prompts.json', prompts)
generations = []
for prompt_id, p in enumerate(prompts):
    for decoding in ('greedy', 'sampling'):
        result = generate(model, p['ids'], decoding=decoding, max_new_tokens=50,
                          temperature=0.8, top_k=50, seed=123, stop_at_eos=True)
        text = ' '.join(id_to_token[i] for i in result['token_ids'])
        row = {'run_id': RUN_ID, 'checkpoint_sha256': MODEL_HASH, 'prompt_id': prompt_id,
               'prompt': p['text'], 'prompt_ids': p['ids'], 'decoding': decoding,
               'temperature': 0.8 if decoding == 'sampling' else None,
               'top_k': 50 if decoding == 'sampling' else None,
               'seed': 123 if decoding == 'sampling' else None,
               'max_new_tokens': 50, 'forbidden_ids': [PAD, UNK, BOS],
               'stop_at_eos': True, **result, 'text': text}
        generations.append(row)
        print(f"\n[{decoding}] {p['text']} → {text}")
with (RESULT_DIR / 'generation.jsonl').open('w', encoding='utf-8') as f:
    for row in generations:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')


[greedy] the history of → the first time , the first time , the first time , the first time , the first time , the first time , the first time , the first time , the first time , the first time , the first time , the first time , the first

[sampling] the history of → the first time , the world by the american tribes and the national park service camps since the . 69 th century . the game ' s to 1981 , foca would represent who has been built within the next day . <EOS>

[greedy] the city is → a " . <EOS>

[sampling] the city is → referred to as the first single mounts for a strong sense of evil " from the old english language . the song . " <EOS>

[greedy] during the first → time , the first time , the first time , the first time , the first time , the first time , the first time , the first time , the first time , the first time , the first time , the first time , the first time ,

[sampling] during the first → in the second @ - @ 4 , ln @ - @ year @ - @ 2 @ , @ - @ - @ - @ - @ based

## 11. Common autoregressive inference benchmark

In [23]:
workload = {
    'prompt_texts': [p['text'] for p in prompts], 'prompt_ids': [p['ids'] for p in prompts],
    'vocab_hash': VOCAB_HASH, 'prompt_file_sha256': completion['sha256']['generation_prompts.json'],
    'batch_size': 1, 'generated_tokens_per_prompt': 50, 'decoding': 'greedy',
    'stop_at_eos': False, 'forbidden_ids': [PAD, UNK, BOS], 'bos_prefix': True,
    'tie_break': 'lowest token ID', 'neural_context_limit': common['block_length'],
    'neural_cache_policy': 'recompute prefix; no KV cache or persistent recurrent state',
    'ngram_context_limit': 2, 'ngram_cache_policy': 'none', 'CPU_threads': CPU_THREADS,
    'warmups': 1, 'repeats': 3,
    'timed_scope': 'probability computation and token selection; exclude loading/tokenization/text/I-O'
}
WORKLOAD_ID = object_hash(workload)
write_json(RUN_DIR / 'inference_workload.json', {'workload_id': WORKLOAD_ID, **workload})
benchmark_rows = []
def benchmark_once():
    count = 0
    for p in prompts:
        count += generate(model, p['ids'], max_new_tokens=50, decoding='greedy', stop_at_eos=False)['generated_length']
    return count
if RUN_BENCHMARK:
    benchmark_once()
    for repeat in range(1, 4):
        start = time.perf_counter()
        count = benchmark_once()
        seconds = time.perf_counter() - start
        assert count == len(prompts)*50
        benchmark_rows.append({'run_id': RUN_ID, 'session_id': SESSION_ID,
            'checkpoint_sha256': MODEL_HASH, 'workload_id': WORKLOAD_ID, 'device': 'cpu',
            'cpu_model': environment['cpu_model'], 'CPU_threads': CPU_THREADS,
            'precision': 'float64', 'cache_policy': 'none', 'prompt_count': len(prompts),
            'generated_tokens': count, 'repeat': repeat, 'seconds': seconds,
            'tokens_per_second': count/seconds})
    benchmark = pd.DataFrame(benchmark_rows)
    benchmark.to_csv(RESULT_DIR / 'benchmark.csv', index=False)
    benchmark_summary = {'workload_id': WORKLOAD_ID, 'session_id': SESSION_ID,
        'median_seconds': float(benchmark.seconds.median()),
        'min_seconds': float(benchmark.seconds.min()), 'max_seconds': float(benchmark.seconds.max()),
        'median_tokens_per_second': float(benchmark.tokens_per_second.median()),
        'min_tokens_per_second': float(benchmark.tokens_per_second.min()),
        'max_tokens_per_second': float(benchmark.tokens_per_second.max())}
    write_json(RESULT_DIR / 'benchmark_summary.json', benchmark_summary)
    display(benchmark[['repeat', 'generated_tokens', 'seconds', 'tokens_per_second']].round(4))
    print('Median and range:', benchmark_summary)
else:
    print('Benchmark skipped by RUN_BENCHMARK=False.')

,repeat,generated_tokens,seconds,tokens_per_second
0,1,200,0.3143,636.3380
1,2,200,0.3335,599.6280
2,3,200,0.3205,623.9893


Median and range: {'workload_id': 'cc76ff95e65571a9d42d5dc86746e3d78f02ae8cfb3084f56d1e1fd182a85ccf', 'session_id': '20261014T154830Z_185bd5', 'median_seconds': 0.32051830299997164, 'min_seconds': 0.31429838799999743, 'max_seconds': 0.33354010300001846, 'median_tokens_per_second': 623.9893264379904, 'min_tokens_per_second': 599.6280453267982, 'max_tokens_per_second': 636.3379757455251}


## 12. Export comparison evidence and a reload guide


In [24]:
entries = model.entry_counts()
test_row = next((r for r in metric_rows if r['split'] == 'test'), None)
summary_row = {
    'run_id': RUN_ID, 'part': 'I', 'seed': SEED, 'checkpoint': str(INFERENCE_PATH),
    'checkpoint_sha256': file_hash(INFERENCE_PATH), 'data_hash': DATA_HASH, 'vocab_hash': VOCAB_HASH,
    'test_nll_sum': test_row['nll_sum'] if test_row else None,
    'test_token_count': test_row['token_count'] if test_row else None,
    'test_ce': test_row['ce'] if test_row else None, 'test_ppl': test_row['ppl'] if test_row else None,
    'test_target_ids_sha256': test_row['target_ids_sha256'] if test_row else None,
    'best_epoch': None, 'epochs_completed': None, 'total_parameters': None,
    'trainable_parameters': None, 'optimizer_updates': None,
    'not_applicable_reason': 'count model; one fitting pass, no neural parameters or epochs',
    'count_entries': sum(entries[k] for k in ['unigram_entries','bigram_entries','trigram_entries']),
    **entries, 'weights_mib': INFERENCE_BYTES / 2**20, 'inference_package_mib': PACKAGE_BYTES / 2**20,
    'train_seconds': fit_info['seconds'], 'validation_seconds': selection_info['seconds'],
    'fit_plus_selection_seconds': fit_info['seconds'] + selection_info['seconds'],
    'train_targets': model.total, 'train_blocks': len(blocks['train'][0]),
    'train_rows': int(data_summary.loc[data_summary['split']=='train', 'retained_rows'].iloc[0]),
    'training_passes': 1, 'block_length': common['block_length'],
    'validation_candidate_count': selection_info['candidate_count'],
    'selected_k': model.k, 'selected_weights': json.dumps(model.weights),
    'device': 'cpu', 'CPU_threads': CPU_THREADS, 'precision': 'float64',
    'training_cpu': fit_info['training_environment']['cpu_model'],
    'training_session_id': fit_info['training_environment']['session_id'],
    'training_peak_rss_mib': fit_info['sampled_peak_rss_mib'],
    'unknown_pct_test_lexical': float(data_summary.loc[data_summary['split']=='test', 'unknown_pct'].iloc[0]),
    'workload_id': WORKLOAD_ID, 'test_status': 'evaluated' if test_row else 'not_requested'
}
pd.DataFrame([summary_row]).to_csv(RESULT_DIR / 'lm_metrics.csv', index=False)
write_json(RESULT_DIR / 'lm_metrics.json', summary_row)
pd.DataFrame(io_log).to_csv(SESSION_DIR / 'checkpoint_io.csv', index=False)
shutil.copy2(SESSION_DIR / 'environment.json', RESULT_DIR / 'environment.json')
shutil.copy2(SESSION_DIR / 'target_audit.json', RESULT_DIR / 'target_audit.json')
shutil.copy2(RUN_DIR / 'validation_search.csv', RESULT_DIR / 'validation_search.csv')
write_json(SESSION_DIR / 'session_status.json', {
    'run_id': RUN_ID, 'completed_utc': datetime.now(timezone.utc).isoformat(),
    'elapsed_since_bootstrap_seconds': time.perf_counter()-SESSION_START,
    'elapsed_note': 'Includes executed-cell work and any pauses between cells; excludes pip install and Drive mount.',
    'test_requested': EVALUATE_TEST, 'benchmark_requested': RUN_BENCHMARK,
    'result_directory': str(RESULT_DIR)})
reload_guide = f"""# Reload {RUN_ID} without training

Use the same Python/NumPy versions recorded in the original session where possible.
Install numpy pandas psutil threadpoolctl; mount Drive and keep the complete run folder.
Load only your own trusted pickle files. Verify inference.pkl against inference.sha256.json.

```python
import sys, json, pickle, hashlib
from pathlib import Path
run_dir = Path({str(RUN_DIR)!r})
sys.path.insert(0, str(run_dir))
import {MODULE_NAME} as core
path = run_dir / 'inference.pkl'
expected = json.loads((run_dir / 'inference.sha256.json').read_text())['sha256']
assert hashlib.sha256(path.read_bytes()).hexdigest() == expected
with path.open('rb') as f:
    package = pickle.load(f)
model = package['model']
# Pass saved x/y blocks for scoring. Do not rebuild their vocabulary.
# counts, totals = model.counts_for_targets(x, y)
# metrics = model.score_counts(counts, totals)
# output = core.generate(model, prompt_ids, decoding='greedy')
```

For exact evaluation provenance reuse common.json, data_complete.json and the
original processed files; verify their checksums and scored target hash.
Use inference_workload.json for the same generation benchmark in notebook 03/05.
N-gram counts use one-token history at block starts and at most two elsewhere.
Count fitting is one training pass; no neural parameter count or learning curve exists.
The four saved prompts are preserved from notebook 01 (200 benchmark tokens/repeat).

Training timing and its machine live in training_cost.json; current session
hardware lives in sessions/<session>/environment.json. Never relabel load time as
training time. Compare CPU inference on one shared session; GPU results separately.

Sources: supplied Assignment1_2026(1).pdf (Part I, reproducibility/tool use),
DSA4213_Assignment1_Implementation_Plan.pdf (Sections 6, 8–11, 14–15), and
01_data_preparation.ipynb (file/schema/tokenizer contract). Dataset provenance
is preserved in data_source.json. This implementation was prepared with
ChatGPT assistance; review and acknowledge substantial assistance in submission.
"""
(RUN_DIR / 'README_ngram.md').write_text(reload_guide, encoding='utf-8')
write_json(RESULT_DIR / 'artifact_manifest.json', {
    'run_id': RUN_ID, 'session_id': SESSION_ID,
    'sha256': {p.name: file_hash(p) for p in sorted(RESULT_DIR.iterdir()) if p.is_file()}})
print('N-GRAM WORKFLOW COMPLETE')
print('Saved model and reload instructions:', RUN_DIR)
print('This session results:', RESULT_DIR)
print('Save your executed notebook in Drive as well.')
display(pd.DataFrame([summary_row])[['run_id','test_ce','test_ppl','count_entries',
    'weights_mib','train_seconds','validation_seconds','fit_plus_selection_seconds']].round(5))

N-GRAM WORKFLOW COMPLETE
Saved model and reload instructions: /content/drive/MyDrive/DSA4213_Assignment1/runs/I_ngram_05cfd29bc588
This session results: /content/drive/MyDrive/DSA4213_Assignment1/results/I_ngram_05cfd29bc588/20261014T154830Z_185bd5
Save your executed notebook in Drive as well.


,run_id,test_ce,test_ppl,count_entries,weights_mib,train_seconds,validation_seconds,fit_plus_selection_seconds
0,I_ngram_05cfd29bc588,4.93253,138.73035,1624963,16.91644,15.85008,1.24553,17.09561
